# FTS ML Hyperparameter Optimization & Model Registry Workspace

Welcome to the interactive hyperparameter optimization and model registry workspace! This notebook shows you how to:
1. **Load database connections and settings** for FTS.
2. **Dynamically explore available model types** and configurations in the registry.
3. **Programmatically override and validate search configurations** before running optimization.
4. **Run Optuna hyperparameter search** using the core `hparam_search` engine.
5. **Visualize search results** using native Optuna Plotly graphs.
6. **Inspect candidate models** in a pandas DataFrame, and **promote** the best model to production.

### 1. Import Dependencies and Initialize DB Connections

In [ ]:
import os
import yaml
import pandas as pd
import optuna
from datetime import datetime, timezone
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from trading_bot.config import settings
from trading_bot.core.database import init_db, SessionLocal
from trading_bot.core.repository import MarketDataRepository, ModelRepository
from trading_bot.core.schemas import BarData

from nets.training.hparam_search import run_hparam_search, TRAINER_REGISTRY
from nets.models import NNTrainingConfig

# Set database URL dynamically to local dev.db
settings.DATABASE_URL = "sqlite+pysqlite:///../dev.db"
db_url = "sqlite:///../dev.db"
engine = create_engine(db_url, pool_pre_ping=True)
SessionLocal.configure(bind=engine)

# Initialize tables
init_db(extra_models=["trading_bot.core.models"], bind_engine=engine)
print("Database engine initialized. Available schemas prepared.")

### 2. Dynamic Model Registry Exploration

To adhere to the Open/Closed Principle (OCP), we retrieve the available model architectures dynamically from the codebase's central registry map. Adding any new model trainer or configuration in the core code automatically registers and exposes it here.

In [ ]:
available_models = list(TRAINER_REGISTRY.keys())
print("Available model types in TRAINER_REGISTRY:", available_models)

for model_type, (trainer_cls, config_cls) in TRAINER_REGISTRY.items():
    print(f"\n- Model Type: '{model_type}'")
    print(f"  Trainer: {trainer_cls.__name__} & Config: {config_cls.__name__}")
    print(f"  Configurable Hyperparameters: {list(config_cls.model_fields.keys())}")

### 3. Load & Override Search Configurations

We load the YAML search configuration file `configs/hparam_search.yaml` programmatically into a Python dictionary. This lets you inspect the configuration and perform overrides directly in cell code for full reproducibility, bypassing complex Jupyter UI states.

In [ ]:
CHOSEN_MODEL_TYPE = "lstm"

config_path = f"../configs/train/BTCUSDT/{CHOSEN_MODEL_TYPE}_hparam_search.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Current YAML Configuration:")
print(yaml.dump(config, default_flow_style=False))

### 4. Configuration Validation

We use the Pydantic schemas (LSP/ISP validation) to dynamically validate that all hyperparameter keys specified under `search_space` match expected parameter fields in the model configuration class and core trainer configurations.

In [ ]:
# Validate loaded configurations dynamically
model_type = config["model_type"]
if model_type not in TRAINER_REGISTRY:
    raise ValueError(f"Model type '{model_type}' is invalid. Supported: {available_models}")

trainer_cls, config_cls = TRAINER_REGISTRY[model_type]
print(f"Validating search space params against {config_cls.__name__} & NNTrainingConfig...")

model_fields = set(config_cls.model_fields.keys())
nn_fields = set(NNTrainingConfig.model_fields.keys())
all_valid_fields = model_fields.union(nn_fields)

search_space = config.get("search_space") or {}
for param in search_space:
    if param not in all_valid_fields:
        print(f"⚠️  WARNING: Parameter '{param}' is not defined in the core model configuration classes.")
    else:
        print(f"  - Parameter '{param}' validated successfully.")

### 4.1. Visualize Raw and Preprocessed Training Data

We load the historical bar data from the database using the same repository queries that the training pipeline runs, apply the log return preprocessing, and visualize both the raw price series and the stationary log returns input features.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from trading_bot.core.dataset import DatasetBuilder
from trading_bot.core.transforms import LogReturnTransform
import numpy as np

# 1. Fetch raw bar data using core repository
market_id = config["market_id"]
interval = config.get("interval", "30m")

with SessionLocal() as db:
    market_repo = MarketDataRepository(db)
    raw_bars = market_repo.get_bars(market_id, interval=interval)

# 2. Convert to DataFrame
df_viz = pd.DataFrame([{
    "timestamp": b.timestamp,
    "open": b.open,
    "high": b.high,
    "low": b.low,
    "close": b.close,
    "volume": b.volume
} for b in raw_bars])

transform = LogReturnTransform()
matrix_viz = df_viz[["close"]].values
returns = transform.transform(matrix_viz)
df_viz["log_return"] = np.insert(returns, 0, np.nan, axis=0)


# 4. Plot original and transformed data using Plotly
fig_viz = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=(f"Raw Close Price ({market_id})", "Transformed Log Returns (Stationary Input)")
)

fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["close"], name="Close Price", line=dict(color="#2196F3")), row=1, col=1)
fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["log_return"], name="Log Return", line=dict(color="#FF9800")), row=2, col=1)

fig_viz.update_layout(height=500, title_text=f"Training Data Overview: {market_id} ({interval})", showlegend=False)
fig_viz.show()

### 5. Execute Hyperparameter Optimization

We invoke `run_hparam_search` directly using our updated config file. Optuna will evaluate different parameter sets, train candidates, log metrics to TensorBoard, and register model metadata to our registry.

In [ ]:
print(f"Starting Optuna study '{config['study_name']}' ({config['n_trials']} trials)...\n")
run_hparam_search(config_path)
print("\nHyperparameter optimization study complete!")

### 6. Plot Optuna Optimization Visualizations

We load the Optuna study programmatically from SQLite and render native interactive plots using Optuna's native Plotly backend.

In [ ]:
optuna_storage = settings.DATABASE_URL.replace("sqlite+pysqlite://", "sqlite://")
try:
    study = optuna.load_study(study_name=config["study_name"], storage=optuna_storage)
    print(f"Loaded study '{study.study_name}' containing {len(study.trials)} trials.")
    print(f"Best trial: {study.best_trial.number} | Best Value: {study.best_value}")

    # Render native Plotly plots
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.show()

    if len(study.trials) > 1:
        fig2 = optuna.visualization.plot_param_importances(study)
        fig2.show()
        
        fig3 = optuna.visualization.plot_slice(study)
        fig3.show()
except Exception as e:
    print("Could not load or plot Optuna study visualizations:", e)

### 7. Inspect Model Registry Candidates

We use pandas to fetch all registered candidates from our database model registry table (`model_registry`). This provides a tabular dashboard of all run trials, hyperparameters, and validation metrics.

In [ ]:
from nets.training.hparam_search import get_scored_models

with SessionLocal() as db:
    df_showcase = get_scored_models(
        db,
        model_type=config["model_type"],
        market_id=config["market_id"],
        interval=config.get("interval", "30m")
    )

# Display showcase dataframe
display_cols = [
    "model_id", "model_type", "market_id", "interval", 
    "val_loss", "ic", "directional_accuracy", "composite_score", "status", "created_at"
]
df_showcase[display_cols].head(15)

### 8. Programmatic Model Promotion

To promote a model to production status, copy the `model_id` from the DataFrame above and paste it below. The repository will demote any active production model sharing the same signature and promote the selected candidate.

In [ ]:
# --- ENTER THE MODEL ID TO PROMOTE ---
model_id_to_promote = ""  # e.g., "model_lstm_btc_usd_..."

if model_id_to_promote:
    with SessionLocal() as db:
        repo = ModelRepository(db)
        repo.promote_to_production(model_id_to_promote)
        db.commit()
    print(f"Model '{model_id_to_promote}' successfully promoted to PRODUCTION status.")
    
    # Display status verification
    with SessionLocal() as db:
        df_verify = pd.read_sql(f"SELECT model_id, status, onnx_path FROM model_registry WHERE model_id='{model_id_to_promote}'", db.bind)
    print("\nUpdated database status:")
    print(df_verify)
else:
    print("Please copy/paste a valid 'model_id' into 'model_id_to_promote' to promote it.")